# Orthogonal and Preconditioned Matrix Optimisers (C04 Topic 04)

**Citations:** Gupta et al. (2018 Shampoo); Zhao et al. (2024 GaLore); Vyas et al. (2024 SOAP); Jordan et al. (2024 Muon).

### Purpose & Topic Bridges
Standard first-order optimisers (AdamW) treat weights as unstructured 1D vectors, ignoring the intrinsic 2D linear operator geometry and singular value spectra of neural network layers. Full second-order optimization (natural gradient, full Hessian) requires intractable $O(N^2)$ memory.
**Muon** projects momentum matrices directly onto the Stiefel manifold of semi-orthogonal matrices ($O = UV^T$) via fast quintic Newton-Schulz iterations, achieving target validation loss in ~35% fewer training steps via BF16 GEMMs. **Shampoo** maintains Kronecker-factored left and right second-moment covariances, while **GaLore** tracks momentum within a compact low-rank projection subspace.

**Decision Boundary:** Deploy **Muon** for internal 2D hidden matrix layers ($W \in \mathbb{R}^{m \times n}$ where $m, n \ge 16$) in Transformers and modern LLMs. When NOT to use: Do NOT apply Muon to 1D vectors, LayerNorm weights, biases, token embeddings, or the output language model head; always pair Muon with an auxiliary AdamW optimizer for non-hidden parameters.

## 2. Mathematical Foundations & Governing Equations

1. **Newton-Schulz Quintic Orthogonalization (Muon):**
$$X_0 = \frac{M}{\|M\|_F + \epsilon}, \quad A_k = X_k X_k^T, \quad X_{k+1} = a X_k + b A_k X_k + c A_k^2 X_k$$
Where $(a, b, c) = (3.4445, -4.7750, 2.0315)$. Then $W_{t+1} = W_t (1 - \eta \lambda) - \eta X_K$.
2. **Shampoo 2D Curvature Preconditioning:**
$$L_t = \beta_2 L_{t-1} + (1-\beta_2) G G^T, \quad R_t = \beta_2 R_{t-1} + (1-\beta_2) G^T G$$
$$\tilde{G}_t = L_t^{-1/4} G_t R_t^{-1/4}, \quad W_{t+1} = W_t (1 - \eta \lambda) - \eta \tilde{G}_t$$
3. **GaLore Low-Rank Gradient Projection:**
$$G_{\text{low}} = P^T G_t, \quad U_{\text{low}} = \text{Adam}(G_{\text{low}}), \quad W_{t+1} = W_t (1 - \eta \lambda) - \eta P U_{\text{low}}$$
4. **Computational Complexity:** Muon requires $\mathcal{O}(m^2 n + m n^2)$ FLOPs per step (5 GEMMs) and $\mathcal{O}(mn)$ memory ($1\times$ momentum buffer, zero variance buffers). Shampoo requires $\mathcal{O}(m^3 + n^3)$ periodic matrix roots and $\mathcal{O}(m^2 + n^2)$ covariance memory. GaLore operates in $\mathcal{O}(rn)$ compact memory.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/04_orthogonal_and_preconditioned_matrix_optimisers.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class OrthogonalMatrixOptimisersUnifiedModule(nn.Module):
    def __init__(self, in_dim=8, out_dim=8):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def newton_schulz(self, G, steps=5, eps=1e-7):
        a, b, c = 3.4445, -4.7750, 2.0315
        m, n = G.shape; X = G / (torch.norm(G) + eps)
        if m >= n: X = X.t()
        for _ in range(steps):
            A = X @ X.t(); B = b * A + c * (A @ A); X = a * X + B @ X
        if m >= n: X = X.t()
        return X

    def step_muon(self, param, grad, m_buf, lr=0.02, momentum=0.95, nesterov=True, weight_decay=0.01, ns_steps=5):
        new_m = momentum * m_buf + grad
        cand = momentum * new_m + grad if nesterov else new_m
        ortho = self.newton_schulz(cand, steps=ns_steps)
        return param * (1.0 - lr * weight_decay) - lr * ortho, new_m

    def step_shampoo(self, param, grad, l_cov, r_cov, lr=0.005, beta2=0.95, eps=1e-6, weight_decay=0.01):
        new_l = beta2 * l_cov + (1.0 - beta2) * (grad @ grad.t())
        new_r = beta2 * r_cov + (1.0 - beta2) * (grad.t() @ grad)
        eval_l, evec_l = torch.linalg.eigh(new_l); l_inv = evec_l @ torch.diag(torch.clamp(eval_l, min=eps)**(-0.25)) @ evec_l.t()
        eval_r, evec_r = torch.linalg.eigh(new_r); r_inv = evec_r @ torch.diag(torch.clamp(eval_r, min=eps)**(-0.25)) @ evec_r.t()
        precond = l_inv @ grad @ r_inv
        return param * (1.0 - lr * weight_decay) - lr * precond, new_l, new_r

    def step_galore(self, param, grad, m_low, v_low, proj, step=2, lr=0.001, b1=0.9, b2=0.999, eps=1e-8, weight_decay=0.01):
        g_low = proj.t() @ grad; new_m = b1 * m_low + (1.0 - b1) * g_low; new_v = b2 * v_low + (1.0 - b2) * (g_low * g_low)
        m_hat = new_m / (1.0 - b1**step); v_hat = new_v / (1.0 - b2**step)
        full_u = proj @ (m_hat / (torch.sqrt(v_hat) + eps))
        return param * (1.0 - lr * weight_decay) - lr * full_u, new_m, new_v

model = OrthogonalMatrixOptimisersUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].to(device).t())
    model.linear.bias.copy_(weights['bias'].to(device))
print('OrthogonalMatrixOptimisersUnifiedModule initialized.')

In [ ]:
p_2d = weights['param_2d'].to(device); g_2d = weights['grad_2d'].to(device)

# 1. Muon Verification
p_muon, m_muon = model.step_muon(p_2d, g_2d, weights['m_init_muon'].to(device))
err_p_muon = (p_muon - weights['p_muon'].to(device)).abs().max().item()
err_m_muon = (m_muon - weights['m_muon'].to(device)).abs().max().item()

# 2. Shampoo Verification
p_shampoo, l_shampoo, r_shampoo = model.step_shampoo(
    p_2d, g_2d, weights['l_init'].to(device), weights['r_init'].to(device)
)
err_p_shampoo = (p_shampoo - weights['p_shampoo'].to(device)).abs().max().item()
err_l_shampoo = (l_shampoo - weights['l_shampoo'].to(device)).abs().max().item()
err_r_shampoo = (r_shampoo - weights['r_shampoo'].to(device)).abs().max().item()

# 3. GaLore Verification
p_galore, m_galore, v_galore = model.step_galore(
    p_2d, g_2d, weights['m_init_galore'].to(device), weights['v_init_galore'].to(device), weights['proj_matrix'].to(device)
)
err_p_galore = (p_galore - weights['p_galore'].to(device)).abs().max().item()
err_m_galore = (m_galore - weights['m_galore'].to(device)).abs().max().item()
err_v_galore = (v_galore - weights['v_galore'].to(device)).abs().max().item()

# 4. Linear Anchor
err_anchor = (model(test_in) - test_out).abs().max().item()

assert max(err_p_muon, err_m_muon) < 1e-5, f'Muon error {err_p_muon:.2e} exceeds 1e-5'
assert max(err_p_shampoo, err_l_shampoo, err_r_shampoo) < 1e-5, f'Shampoo error {err_p_shampoo:.2e} exceeds 1e-5'
assert max(err_p_galore, err_m_galore, err_v_galore) < 1e-5, f'GaLore error {err_p_galore:.2e} exceeds 1e-5'
assert err_anchor < 1e-5, f'Anchor error {err_anchor:.2e} exceeds 1e-5'
print(f'All 4 invariants verified! Muon: {err_p_muon:.2e}, Shampoo: {err_p_shampoo:.2e}, GaLore: {err_p_galore:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'Muon 2D Step': lambda: model.step_muon(p_2d, g_2d, weights['m_init_muon'].to(device)),
    'Shampoo Precond': lambda: model.step_shampoo(p_2d, g_2d, weights['l_init'].to(device), weights['r_init'].to(device)),
    'GaLore Low-Rank': lambda: model.step_galore(p_2d, g_2d, weights['m_init_galore'].to(device), weights['v_init_galore'].to(device), weights['proj_matrix'].to(device)),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=8)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Applying Muon to 1D Parameters vs AdamW Hybrid Routing
# Muon assumes 2D matrix geometry with balanced aspect ratio.
# On 1D normalization channels with non-uniform gradients, 1D Muon cannot adapt per-channel variance:
grad_asym = torch.tensor([0.001, 0.010, 0.500, 0.005], device=device)

# 1. 1D Muon treats the layer as a monolithic vector, preserving raw gradient imbalance:
norm_g = torch.norm(grad_asym)
muon_step = grad_asym / norm_g
muon_ratio = float(muon_step[0] / muon_step[2])  # 0.001 / 0.500 = 0.002

# 2. AdamW adapts coordinate-wise, scaling each channel by its individual noise floor:
adam_step = torch.sign(grad_asym)  # Coordinate-wise adaptive updates step uniformly
adam_ratio = float(adam_step[0] / adam_step[2])  # 1.0

print(f'[FAILURE DEMONSTRATED] 1D Muon unadapted coordinate ratio: {muon_ratio:.4f} (inactive channel starved)')
print(f'[CORRECT] Hybrid architecture routing with AdamW adapts coordinates independently: ratio = {adam_ratio:.1f}')
assert muon_ratio < 0.01, 'Muon did not preserve raw gradient imbalance on 1D vector'
assert adam_ratio == 1.0, 'AdamW failed to adapt coordinate-wise'
print('[DIAGNOSTIC] Muon must only be applied to 2D hidden matrices; 1D norms and embeddings must use AdamW.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Matrix Geometry Matters:** Elementwise optimisers (AdamW) ignore tensor structure. Muon equalizes singular values via quintic Newton-Schulz iterations, achieving superior convergence via pure BF16 tensor core GEMMs.
2. **Hybrid Optimizer Architecture is Mandatory:** Never apply Muon globally. High-performing pretraining setups pair Muon on 2D hidden weights with decoupled AdamW on token embeddings, LayerNorms, and LM heads.
3. **Curvature Isomorphism:** Shampoo approximates Kronecker-factored Fisher curvature $L^{-1/4} G R^{-1/4}$, mirroring 2D separable spatial filter design in signal processing and Riemannian Stiefel geodesics.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 03_large_batch_and_layerwise_optimisers](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/03_large_batch_and_layerwise_optimisers.ipynb) | [Next: 05_second_order_and_curvature_optimisers](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/05_second_order_and_curvature_optimisers.ipynb)